In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("Gemini_same_question_response_final.pkl", "rb") as f:
    Gemini_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    if model=="gemini-2.5-flash-lite":
        dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]
    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 350
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: gemini-2.5-flash-lite


##### Associate id with all gene entry

In [8]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 17:51:50,086 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:51:50,087 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:51:50,088 [INFO] querying 1-940 ...
2026-02-27 17:51:52,460 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 17:51:53,466 [INFO] Finished.
2026-02-27 17:51:53,475 [WARNING] 2 input query terms found dup hits:	[('IGHG1', 2), ('MTOR', 2)]
2026-02-27 17:51:53,475 [WARNING] 182 input query terms found no hit:	['*/;', '...', '</H3>', 'ALK FUSION', 'ARGHECN1', 'ARGHEIS', 'ARPP', 'ARPP-21', 'AS', 'BCR-ABL1', 'B
2026-02-27 17:51:53,489 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:51:53,489 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 17:51:53,490 [INFO] querying 1-53 ...
2026-02-27 17:51:53,861 [INFO] HTTP Request: POST htt

[warn] Could not resolve '*/;' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve '...' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'ALK fusion' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'ARGHECN1' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'ARGHEIS' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'BCR-ABL1' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'BRAF V600E' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'C1QTNFIP3' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'CD3' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not reso

,gene_name,gene_id,source
0,*/;,NaN,None
1,...,NaN,None
2,</h3>,ENSG00000101180,mygene
3,ABCA7,ENSG00000064687,mygene
4,ABCG2,ENSG00000118777,mygene
...,...,...,...
935,```,NaN,None
936,gấp,NaN,None
937,mTOR,ENSG00000198793,mygene
938,p38,ENSG00000100030,opentargets


In [9]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source
0,*/;,NaN,None
1,...,NaN,None
16,ALK fusion,NaN,None
41,ARGHECN1,NaN,None
42,ARGHEIS,NaN,None
...,...,...,...
917,ZDHHC10,NaN,None
926,ZNF436,NaN,None
934,_G6PC,NaN,None
935,```,NaN,None


##### Associate id with all drug entry

In [10]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 17:54:09,738 [INFO] [LLM] Total 80 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:54:09,739 [INFO] [LLM] Batch size: 80


[map][drug][OpenTargets] raw='5-fluorouracil' -> id='CHEMBL185'
[map][drug][OpenTargets] raw='Abraxane' -> id='CHEMBL428647'
[map][drug][OpenTargets] raw='Ado-trastuzumab emtansine' -> id='CHEMBL1743082'
[map][drug][OpenTargets] raw='Afatinib' -> id='CHEMBL1173655'
[map][drug][OpenTargets] raw='Aimovig' -> id='CHEMBL3833329'
[map][drug][OpenTargets] raw='Ajovy' -> id='CHEMBL4297756'
[map][drug][OpenTargets] raw='Alfacalcidol' -> id='CHEMBL1601669'
[map][drug][OpenTargets] raw='Almotriptan' -> id='CHEMBL1505'
[map][drug][OpenTargets] raw='Amantadine' -> id='CHEMBL660'
[map][drug][OpenTargets] raw='Amivantamab' -> id='CHEMBL4297774'
[map][drug][OpenTargets] raw='Apomorphine' -> id='CHEMBL53'
[map][drug][OpenTargets] raw='Asciminib' -> id='CHEMBL4208229'
[map][drug][OpenTargets] raw='Atezolizumab' -> id='CHEMBL3707227'
[map][drug][OpenTargets] raw='Axicabtagene ciloleucel' -> id='CHEMBL3989989'
[map][drug][OpenTargets] raw='Axitinib' -> id='CHEMBL1289926'
[map][drug][OpenTargets] raw='Aza

2026-02-27 17:54:11,690 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:54:11,700 [INFO] [LLM] Batch done in 1.96s
2026-02-27 17:54:11,701 [INFO] [OT] Verifying 19 unique drug suggestions...
2026-02-27 17:54:12,146 [INFO] [OT] 'tocilizumab' → 'TOCILIZUMAB' (CHEMBL1237022)
2026-02-27 17:54:12,147 [INFO] [OT] 'upadacitinib' → 'UPADACITINIB' (CHEMBL3622821)
2026-02-27 17:54:12,148 [INFO] [OT] 'irinotecan' → 'IRINOTECAN' (CHEMBL481)
2026-02-27 17:54:12,148 [INFO] [OT] 'leflunomide' → 'LEFLUNOMIDE' (CHEMBL960)
2026-02-27 17:54:12,149 [INFO] [OT] 'paclitaxel' → 'PACLITAXEL' (CHEMBL428647)
2026-02-27 17:54:12,149 [INFO] [OT] 'sulfasalazine' → 'SULFASALAZINE' (CHEMBL421)
2026-02-27 17:54:12,150 [INFO] [OT] 'prednisone' → 'PREDNISONE' (CHEMBL635)
2026-02-27 17:54:12,150 [INFO] [OT] 'rituximab' → 'RITUXIMAB' (CHEMBL1201576)
2026-02-27 17:54:12,150 [INFO] [OT] 'tofacitinib' → 'TOFACITINIB' (CHEMBL221959)
2026-02-27 17:54:12,151 [INFO] [OT]

[map][drug][Llama] raw='adalimumab-afal' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-aflx' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-aqbz' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-bdgm' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-bmsj' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-dyar' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-htkn' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-kmrt' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-lwbs' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-sybb' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] raw='adalimumab-tmcn' -> canonical='ADALIMUMAB' -> id='CHEMBL1201580'
[map][drug][Llama] ra

,drug_name,drug_id,source
0,,None,None
1,5-fluorouracil,CHEMBL185,OpenTargets
2,Abraxane,CHEMBL428647,OpenTargets
3,Ado-trastuzumab emtansine,CHEMBL1743082,OpenTargets
4,Afatinib,CHEMBL1173655,OpenTargets
...,...,...,...
344,venlafaxine,CHEMBL637,OpenTargets
345,verapamil,CHEMBL6966,OpenTargets
346,vinorelbine,CHEMBL553025,OpenTargets
347,z Jucabtagene autoleucel,None,None


In [11]:
df_drug["source"].value_counts()

source
OpenTargets    268
Llama           60
Name: count, dtype: int64

In [12]:
# df_drug[df_drug["source"]=="Llama"]

In [13]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
0,,None,None
22,COMT inhibitors,None,None
32,Dopamine agonists,None,None
42,FOLFIRINOX,None,None
50,Innotyucel,None,None
58,MAO-B inhibitors,None,None
77,Platinum-based chemotherapy,None,None
91,Stalevo,None,None
109,Zinbota,None,None
113,abatacept-kybx,None,None


##### Associate id with all disease entry

In [14]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 17:54:14,940 [INFO] [LLM] Total 43 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:54:14,941 [INFO] [LLM] Batch size: 43


[map][disease][OpenTargets] raw='Achondroplasia' -> id='MONDO_0007037'
[map][disease][OpenTargets] raw='Acute lymphoblastic leukemia' -> id='EFO_0000220'
[map][disease][OpenTargets] raw='Acute myeloid leukemia' -> id='EFO_0000222'
[map][disease][OpenTargets] raw='Adenocarcinoma' -> id='EFO_0000228'
[map][disease][OpenTargets] raw='Alveolar soft part sarcoma' -> id='EFO_0007143'
[map][disease][OpenTargets] raw='Alzheimer's disease' -> id='MONDO_0007088'
[map][disease][OpenTargets] raw='Amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][OpenTargets] raw='Anaplastic large cell lymphoma' -> id='EFO_0003032'
[map][disease][OpenTargets] raw='Angiosarcoma' -> id='EFO_0003968'
[map][disease][OpenTargets] raw='Antley-Bixler syndrome' -> id='MONDO_0008803'
[map][disease][OpenTargets] raw='Apert syndrome' -> id='MONDO_0007041'
[map][disease][OpenTargets] raw='Astrocytoma' -> id='EFO_0000272'
[map][disease][OpenTargets] raw='B-cell lymphoma' -> id='EFO_1001938'
[map][disease][Open

2026-02-27 17:54:16,239 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:54:16,241 [INFO] [LLM] Batch done in 1.30s
2026-02-27 17:54:16,242 [INFO] [LLM] 'Appendiceal cancer' → 'appendix cancer'
2026-02-27 17:54:16,243 [INFO] [LLM] 'Bronchial carcinoid tumor' → 'bronchial carcinoid tumor'
2026-02-27 17:54:16,243 [INFO] [LLM] 'Carcinogenesis' → None
2026-02-27 17:54:16,243 [INFO] [LLM] 'Co-arctation of the aorta' → 'coarctation of the aorta'
2026-02-27 17:54:16,244 [INFO] [LLM] 'Colorectal Neoplasms' → 'colorectal cancer'
2026-02-27 17:54:16,245 [INFO] [LLM] 'Cutaneous anaplastic large cell lymphoma' → 'cutaneous anaplastic large cell lymphoma'
2026-02-27 17:54:16,245 [INFO] [LLM] 'Disease' → None
2026-02-27 17:54:16,246 [INFO] [LLM] 'Fibroblast growth factor receptor 2-related developmental disorder' → None
2026-02-27 17:54:16,246 [INFO] [LLM] 'Gastrointestinal cancer' → 'gastrointestinal cancer'
2026-02-27 17:54:16,246 [INFO] [LL

[map][disease][Llama] raw='Appendiceal cancer' -> canonical='appendix cancer' -> id='MONDO_0001235'
[map][disease][Llama] raw='Co-arctation of the aorta' -> canonical='Aortic Coarctation' -> id='EFO_1001267'
[map][disease][Llama] raw='Colorectal Neoplasms' -> canonical='colorectal carcinoma' -> id='EFO_1001951'
[map][disease][Llama] raw='HER2-low breast cancer' -> canonical='breast carcinoma' -> id='EFO_0000305'
[map][disease][Llama] raw='HER2-mutant lung cancer' -> canonical='lung carcinoma' -> id='EFO_0001071'
[map][disease][Llama] raw='HER2-mutant non-small cell lung cancer' -> canonical='non-small cell lung carcinoma' -> id='EFO_0003060'
[map][disease][Llama] raw='HER2-positive advanced or metastatic breast cancer' -> canonical='breast carcinoma' -> id='EFO_0000305'
[map][disease][Llama] raw='HER2-positive breast cancer' -> canonical='breast carcinoma' -> id='EFO_0000305'
[map][disease][Llama] raw='HER2-positive gastric cancer' -> canonical='stomach neoplasm' -> id='EFO_0003897'
[m

,disease_name,disease_id,source
0,Achondroplasia,MONDO_0007037,OpenTargets
1,Acute lymphoblastic leukemia,EFO_0000220,OpenTargets
2,Acute myeloid leukemia,EFO_0000222,OpenTargets
3,Adenocarcinoma,EFO_0000228,OpenTargets
4,Alveolar soft part sarcoma,EFO_0007143,OpenTargets
...,...,...,...
342,tumor mutational burden-high solid tumors,None,None
343,type 2 diabetes mellitus,MONDO_0005148,OpenTargets
344,ulcerative colitis,EFO_0000729,OpenTargets
345,urothelial carcinoma,EFO_0008528,OpenTargets


In [15]:
df_disease["source"].value_counts()

source
OpenTargets    304
Llama           20
Name: count, dtype: int64

In [16]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
20,Bronchial carcinoid tumor,None,None
23,Carcinogenesis,None,None
49,Cutaneous anaplastic large cell lymphoma,None,None
53,Disease,None,None
63,Fibroblast growth factor receptor 2-related de...,None,None
68,Gastrointestinal cancer,None,None
95,Hutchenreuter-TP53 syndrome,None,None
106,Large cell lymphoma,None,None
145,Multiple types of cancer,None,None
151,Neoplasms,None,None


In [17]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 17:54:17,264 [INFO] [LLM] Total 23 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 17:54:17,264 [INFO] [LLM] Batch size: 23
2026-02-27 17:54:18,146 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 17:54:18,148 [INFO] [LLM] Batch done in 0.88s
2026-02-27 17:54:18,148 [INFO] [LLM] 'Bronchial carcinoid tumor' → 'bronchial carcinoid tumor'
2026-02-27 17:54:18,149 [INFO] [LLM] 'Carcinogenesis' → None
2026-02-27 17:54:18,149 [INFO] [LLM] 'Cutaneous anaplastic large cell lymphoma' → 'cutaneous anaplastic large cell lymphoma'
2026-02-27 17:54:18,150 [INFO] [LLM] 'Disease' → None
2026-02-27 17:54:18,150 [INFO] [LLM] 'Fibroblast growth factor receptor 2-related developmental disorder' → None
2026-02-27 17:54:18,150 [INFO] [LLM] 'Gastrointestinal cancer' → None
2026-02-27 17:54:18,151 [INFO] [LLM] 'Hutchenreuter-TP53 syndrome' → None
2026-02-27 17:54:18,151 [INFO] [LLM

[summary][disease][audit] OT-mappable candidates: 0
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical    23
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,Bronchial carcinoid tumor,None,None,None,no_canonical,False,False
1,Carcinogenesis,None,None,None,no_canonical,False,False
2,Cutaneous anaplastic large cell lymphoma,None,None,None,no_canonical,False,False
3,Disease,None,None,None,no_canonical,False,False
4,Fibroblast growth factor receptor 2-related de...,None,None,None,no_canonical,False,False
5,Gastrointestinal cancer,None,None,None,no_canonical,False,False
6,Hutchenreuter-TP53 syndrome,None,None,None,no_canonical,False,False
7,Large cell lymphoma,None,None,None,no_canonical,False,False
8,Multiple types of cancer,None,None,None,no_canonical,False,False
9,Neoplasms,None,None,None,no_canonical,False,False


In [18]:
# disease_canonical_audit_df

In [19]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
11,Appendiceal cancer,MONDO_0001235,Llama
36,Co-arctation of the aorta,EFO_1001267,Llama
40,Colorectal Neoplasms,EFO_1001951,Llama
74,HER2-low breast cancer,EFO_0000305,Llama
75,HER2-mutant lung cancer,EFO_0001071,Llama
76,HER2-mutant non-small cell lung cancer,EFO_0003060,Llama
77,HER2-positive advanced or metastatic breast ca...,EFO_0000305,Llama
78,HER2-positive breast cancer,EFO_0000305,Llama
79,HER2-positive gastric cancer,EFO_0003897,Llama
80,HER2-positive metastatic breast cancer,EFO_0000305,Llama


In [20]:
df_disease["source"].value_counts()

source
OpenTargets    304
Llama           20
Name: count, dtype: int64

In [21]:
df_disease

,disease_name,disease_id,source
0,Achondroplasia,MONDO_0007037,OpenTargets
1,Acute lymphoblastic leukemia,EFO_0000220,OpenTargets
2,Acute myeloid leukemia,EFO_0000222,OpenTargets
3,Adenocarcinoma,EFO_0000228,OpenTargets
4,Alveolar soft part sarcoma,EFO_0007143,OpenTargets
...,...,...,...
342,tumor mutational burden-high solid tumors,None,None
343,type 2 diabetes mellitus,MONDO_0005148,OpenTargets
344,ulcerative colitis,EFO_0000729,OpenTargets
345,urothelial carcinoma,EFO_0008528,OpenTargets


In [22]:
# df_disease[df_disease["source"]=="Llama"]

In [23]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: gemini-2.5-flash-lite
'Which diseases are associated with BRAF?': J=0.1579, |∩|=3, |∪|=19, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.1429, |∩|=11, |∪|=77, runs=5
'Which diseases are associated with CDK4?': J=0.1500, |∩|=3, |∪|=20, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=0.4583, |∩|=11, |∪|=24, runs=5
'Which genes are associated with ovarian cancer?': J=0.1277, |∩|=6, |∪|=47, runs=5
'What are the known targets of regorafenib?': J=0.0789, |∩|=3, |∪|=38, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.1667, |∩|=1, |∪|=6, runs=5
'Which diseases are treated with bevacizumab?': J=0.3846, |∩|=5, |∪|=13, runs=5
'What are the known targets of cabozantinib?': J=0.6667, |∩|=6, |∪|=9, runs=5
Not enough valid runs for 

In [24]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: gemini-2.5-flash-lite
'Which diseases are associated with BRAF?': J=0.1579, |∩|=3, |∪|=19, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=0.1429, |∩|=11, |∪|=77, runs=5
'Which diseases are associated with CDK4?': J=0.1500, |∩|=3, |∪|=20, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=0.4583, |∩|=11, |∪|=24, runs=5
'Which genes are associated with ovarian cancer?': J=0.1277, |∩|=6, |∪|=47, runs=5
'What are the known targets of regorafenib?': J=0.0789, |∩|=3, |∪|=38, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.1667, |∩|=1, |∪|=6, runs=5
'Which diseases are treated with bevacizumab?': J=0.3846, |∩|=5, |∪|=13, runs=5
'What are the known targets of cabozantinib?': J=0.6667, |∩|=6, |∪|=9, runs=5
Not enough valid runs for 

,model,question,jaccard,n_valid_runs,intersection_size,union_size
9,gemini-2.5-flash-lite,Which diseases are associated with the NPM1 gene?,1.000000,5,1,1
14,gemini-2.5-flash-lite,For which diseases is darolutamide used?,1.000000,5,1,1
19,gemini-2.5-flash-lite,What are the known targets of vandetanib?,1.000000,5,3,3
31,gemini-2.5-flash-lite,Which diseases are associated with HTT?,1.000000,5,1,1
34,gemini-2.5-flash-lite,Which genes are associated with cystic fibrosis?,1.000000,5,1,1
39,gemini-2.5-flash-lite,Which diseases are treated with sorafenib?,1.000000,5,3,3
22,gemini-2.5-flash-lite,Which drugs are used to treat chronic myeloid ...,0.833333,5,5,6
37,gemini-2.5-flash-lite,Which diseases are treated with adalimumab?,0.800000,5,8,10
27,gemini-2.5-flash-lite,Which biological targets does tofacitinib inte...,0.750000,5,3,4
8,gemini-2.5-flash-lite,What are the known targets of cabozantinib?,0.666667,5,6,9


In [25]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("Gemini_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
